# Entregable 3: preprocesamiento sin fuga, métricas y experimentación

Riesgo de los commits del kernel de Linux con un clasificador difuso-evolutivo. Juan Felipe Rodríguez Galindo (20261595004), Inteligencia Computacional, MCIC-UD.

El cuaderno usa la copia incluida en el ZIP o descarga la tabla de commits del [release v1.0](https://github.com/Juferoga/linux-kernel-commits-jit/releases/tag/v1.0), **reserva la prueba (enero a julio de 2026) sin mirarla** y reproduce las tablas de partición, preprocesamiento, espacios de hiperparámetros y validación de las referencias del artículo. El modelo difuso-evolutivo se entrena en el Entregable 4.

In [1]:
URL = ("https://github.com/Juferoga/linux-kernel-commits-jit/releases/download/v1.0/"
       "kernel-commits-2019-2026.parquet")
SHA256 = "ac6858f4b16cb9ee4c7fd1f8baf3c28d480a729dbce9829665d9cdc65045fe9e"
ARCHIVO = "kernel-commits-2019-2026.parquet"
INICIO, HUECO_DIAS = "2019-01-01", 90
VENTANAS_VAL = {"2023": ("2023-01-01", "2024-01-01"), "2024": ("2024-01-01", "2025-01-01"),
                "2025": ("2025-01-01", "2026-01-01")}
PRUEBA = ("2026-01-01", "2026-08-01")
SEMILLA = 11
REJILLAS = {
    "mayoritaria": [{}],
    "logistica": [{"C": c} for c in (0.01, 0.1, 1.0)],
    "gboosting": [{"learning_rate": lr, "max_leaf_nodes": h}
                  for lr in (0.05, 0.1) for h in (15, 31, 63)],
}

In [2]:
import hashlib, json, os, platform, time, urllib.request
import numpy as np, pandas as pd, sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
print("Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)
print("máquina:", platform.machine(), "|", os.cpu_count(), "núcleos")

Python 3.13.15 | pandas 3.1.1 | numpy 2.5.4 | scikit-learn 1.9.1
máquina: arm64 | 8 núcleos


## 1. Descarga y verificación

In [3]:
if not os.path.exists(ARCHIVO):
    local = "datos/juan-rodriguez-20261595004-kernel-commits-2019-2026.parquet"
    if os.path.exists(local):
        ARCHIVO = local
    else:
        urllib.request.urlretrieve(URL, ARCHIVO)
huella = hashlib.sha256(open(ARCHIVO, "rb").read()).hexdigest()
assert huella == SHA256, "el archivo no coincide con el release v1.0"
df = pd.read_parquet(ARCHIVO)
print("SHA-256 verificado |", df.shape[0], "filas x", df.shape[1], "columnas")

SHA-256 verificado | 641926 filas x 33 columnas


## 2. Limpieza y reserva de la prueba

Se quitan los commits censurados (menos de 90 días de observación) y los posteriores al fin de la prueba. De la prueba solo se guardan su tamaño y una huella SHA-256 de sus hashes y etiquetas, que la evaluación final debe reproducir. Después se borra.

In [4]:
def ts(x):
    t = pd.Timestamp(x)
    return t.tz_localize("UTC") if t.tzinfo is None else t.tz_convert("UTC")

tramo = lambda d, a, b: d[(d.fecha >= ts(a)) & (d.fecha < ts(b))]
limpieza = {"filas en la tabla": len(df),
            "hashes duplicados": int(df.hash.duplicated().sum()),
            "censurados (se quitan)": int(df.censurado.sum())}
obs = df[~df.censurado].sort_values("t_commit", kind="stable").reset_index(drop=True)
limpieza["posteriores a julio de 2026 (se quitan)"] = int((obs.fecha >= ts(PRUEBA[1])).sum())
prueba = tramo(obs, *PRUEBA)
h = hashlib.sha256()
for c, r in zip(prueba.hash, prueba.riesgoso):
    h.update(f"{c}{r}".encode())
HUELLA_PRUEBA, N_PRUEBA = h.hexdigest(), len(prueba)
dev = tramo(obs, INICIO, PRUEBA[0]).reset_index(drop=True)
del df, obs, prueba
limpieza["desarrollo 2019-2025"] = len(dev)
limpieza["prueba 2026 (reservada)"] = N_PRUEBA
print("huella de la prueba:", HUELLA_PRUEBA)
pd.Series(limpieza, name="commits").to_frame()

huella de la prueba: 11e7039f3cc279120025ccc8959a207990a4a75f6546ae4c330605c831a19b63


,commits
filas en la tabla,641926
hashes duplicados,0
censurados (se quitan),26133
posteriores a julio de 2026 (se quitan),2982
desarrollo 2019-2025,558438
prueba 2026 (reservada),54373


## 3. Partición temporal con hueco (Tabla de partición)

Ventana creciente: para validar el año $V$ se entrena desde 2019 hasta 90 días antes de que empiece $V$. El hueco imita la práctica: al entrenar, las etiquetas de los últimos 90 días todavía no existen. Nunca se mezclan commits al azar.

In [5]:
def particion(inicio_eval):
    fin = ts(inicio_eval) - pd.Timedelta(days=HUECO_DIAS)
    return tramo(dev, INICIO, fin)

filas = []
for v, (a, b) in VENTANAS_VAL.items():
    tr, va = particion(a), tramo(dev, a, b)
    hueco = (va.fecha.min() - tr.fecha.max()).days
    assert hueco >= HUECO_DIAS - 1 and tr.fecha.max() < va.fecha.min()
    filas.append({"ventana": v, "entrenamiento": f"{INICIO} a {tr.fecha.max():%Y-%m-%d}",
                  "n_entr": len(tr), "prev_entr": tr.riesgoso.mean(),
                  "n_val": len(va), "prev_val": va.riesgoso.mean(), "hueco_dias": hueco})
tabla_particion = pd.DataFrame(filas).set_index("ventana")
tabla_particion.round({"prev_entr": 4, "prev_val": 4})

,entrenamiento,n_entr,prev_entr,n_val,prev_val,hueco_dias
ventana,,,,,,
2023,2019-01-01 a 2022-10-02,300285,0.0439,84780,0.0389,90
2024,2019-01-01 a 2023-10-02,380343,0.0431,76501,0.0414,90
2025,2019-01-01 a 2024-10-02,462363,0.0425,78150,0.0427,90


## 4. Preprocesamiento sin fuga

Las 22 variables son numéricas y se conocen en el momento de integrar el commit. Ninguna entrada se deriva de commits posteriores.

In [6]:
VARIABLES = ["lineas_add", "lineas_del", "n_archivos", "n_binarios", "n_dirs_raiz",
             "n_subsistemas", "n_reviewed", "n_acked", "n_tested", "n_reported", "n_signed",
             "n_codev", "n_link", "cc_stable", "es_fix", "asunto_fix", "len_asunto",
             "exp_autor", "dias_desde_primer_commit", "exp_autor_subsistema", "hora_utc",
             "dia_semana"]
POSTERIORES = {"dias_a_fix", "riesgoso", "censurado"}
assert not POSTERIORES & set(VARIABLES)

s = dev.sort_values("t_autor", kind="stable")
mono = (s.groupby("autor").exp_autor.diff().dropna() >= 0).mean()
print(f"experiencia no decreciente en el tiempo: {mono:.2%} de los pares consecutivos")
BINARIAS = ["cc_stable", "es_fix", "asunto_fix"]
CONTEOS = [v for v in VARIABLES if v not in BINARIAS]
tr25 = particion(VENTANAS_VAL["2025"][0])
z = np.log1p(tr25[CONTEOS].clip(lower=0))
q1, q3 = z.quantile(0.25), z.quantile(0.75)
atipicos = ((z < q1 - 3 * (q3 - q1)) | (z > q3 + 3 * (q3 - q1))).mean()
pasos = pd.DataFrame({
    "faltantes_%": 100 * dev[VARIABLES].isna().mean(),
    "atipicos_%_entr2025": 100 * atipicos.reindex(VARIABLES).fillna(0),
})
pasos.round(2).T

experiencia no decreciente en el tiempo: 99.70% de los pares consecutivos


,lineas_add,lineas_del,n_archivos,n_binarios,n_dirs_raiz,n_subsistemas,n_reviewed,n_acked,n_tested,n_reported,n_signed,n_codev,n_link,cc_stable,es_fix,asunto_fix,len_asunto,exp_autor,dias_desde_primer_commit,exp_autor_subsistema,hora_utc,dia_semana
faltantes_%,0.00,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,0.0
atipicos_%_entr2025,0.02,0.12,1.88,0.0,8.91,9.47,0.0,15.63,6.86,6.4,28.71,1.13,0.0,0.0,0.0,0.0,0.07,0.0,4.23,0.0,2.48,0.0


Cada paso del preprocesamiento queda dentro de un `Pipeline` y se ajusta solo con el entrenamiento de cada ventana:

| Paso | Qué se hace | Dónde se ajusta |
|---|---|---|
| Limpieza | Se quitan los censurados y los posteriores a julio de 2026; no hay hashes duplicados; las copias por *cherry-pick* se conservan (E2, D3) | Una vez, antes de partir |
| Faltantes | No hay (0 %), así que no se imputa | — |
| Atípicos | Las colas son reales (commits enormes), no errores; se conservan y se comprimen con $\log(1+x)$ | Transformación fija, sin parámetros |
| Codificación | No hay categóricas en las entradas (`subsistema` no entra). Hora y día quedan como enteros | — |
| Escalado | `StandardScaler` para la regresión logística; el *boosting* no lo necesita; el difuso usa `QuantileTransformer` | Entrenamiento de cada ventana |
| Selección | Las 22 variables para las referencias y 10 legibles para el difuso, fijadas en el E2 antes de entrenar | Antes de ver resultados |
| Balanceo | No se balancea: la métrica principal (AUC-PR) no depende de un umbral y el umbral del MCC se fija en validación | — |

In [7]:
def log1p_no_negativo(x):
    return np.log1p(np.clip(x, 0, None))

def pipeline(nombre, **hp):
    if nombre == "mayoritaria":
        return Pipeline([("modelo", DummyClassifier(strategy="prior"))])
    if nombre == "logistica":
        return Pipeline([("log1p", FunctionTransformer(log1p_no_negativo)),
                         ("escala", StandardScaler()),
                         ("modelo", LogisticRegression(C=hp["C"], max_iter=3000))])
    if nombre == "gboosting":
        return Pipeline([("modelo", HistGradientBoostingClassifier(
            learning_rate=hp["learning_rate"], max_leaf_nodes=hp["max_leaf_nodes"],
            max_iter=200, random_state=SEMILLA))])
    raise ValueError(nombre)


p = pipeline("logistica", C=1.0).fit(tr25[VARIABLES], tr25.riesgoso)
media_aprendida = p.named_steps["escala"].mean_[0]
print(f"media de log1p(lineas_add): aprendida {media_aprendida:.4f} | entrenamiento {np.log1p(tr25.lineas_add).mean():.4f} | todo el desarrollo {np.log1p(dev.lineas_add).mean():.4f}")
assert np.isclose(media_aprendida, np.log1p(tr25.lineas_add).mean())

media de log1p(lineas_add): aprendida 2.2799 | entrenamiento 2.2799 | todo el desarrollo 2.2937


## 5. Métricas y pruebas, declaradas antes de entrenar el modelo propuesto

- **Principal:** AUC-PR (precisión promedio) de la clase *riesgoso*. Decide qué modelo gana.
- **Secundarias:** MCC, AUC-ROC, sensibilidad y especificidad, y AUC-PR / prevalencia. Las que dependen de un umbral usan el que maximiza el MCC en la validación 2025, aplicado tal cual en la prueba.
- **Incertidumbre:** IC 95 % por *bootstrap* (1000 remuestreos) en la prueba y media ± DE en 5 semillas (11, 23, 37, 41, 53) para los modelos estocásticos.
- **Pruebas:** McNemar con corrección de continuidad entre el difuso-evolutivo y cada referencia, sobre las mismas predicciones de prueba; Friedman con *post hoc* de Nemenyi sobre los rangos de AUC-PR en las tres ventanas más la prueba. α = 0,05. Friedman–Nemenyi se interpreta como exploratorio por el solapamiento del entrenamiento y la selección de hiperparámetros en esas ventanas.
- **Costo:** tiempo de entrenamiento, tiempo de inferencia y número de parámetros.

## 6. Espacios de hiperparámetros

In [8]:
ESPACIO_FE = {"n_reglas": [5, 10, 15], "poblacion": [40, 60, 80], "generaciones": [60, 100, 140], "prob_mutacion": [0.05, 0.15, 0.3], "prob_cruce": [0.7, 0.9], "lambda_complejidad": [0.0, 0.0005]}
FIJOS_FE = {"terminos_por_variable": 3, "max_antecedentes": 3, "torneo": 3, "elitismo": 2, "muestra_aptitud": 80000}
espacios = pd.DataFrame(
    [("logistica", "C", str([r["C"] for r in REJILLAS["logistica"]]), "rejilla completa")] +
    [("gboosting", "learning_rate x max_leaf_nodes", "[0.05, 0.1] x [15, 31, 63]", "rejilla completa")] +
    [("difuso_evolutivo", k, str(v), "aleatoria, 14 configuraciones") for k, v in ESPACIO_FE.items()],
    columns=["modelo", "hiperparámetro", "valores", "búsqueda"])
print("fijos del difuso:", FIJOS_FE)
espacios

fijos del difuso: {'terminos_por_variable': 3, 'max_antecedentes': 3, 'torneo': 3, 'elitismo': 2, 'muestra_aptitud': 80000}


,modelo,hiperparámetro,valores,búsqueda
0,logistica,C,"[0.01, 0.1, 1.0]",rejilla completa
1,gboosting,learning_rate x max_leaf_nodes,"[0.05, 0.1] x [15, 31, 63]",rejilla completa
2,difuso_evolutivo,n_reglas,"[5, 10, 15]","aleatoria, 14 configuraciones"
3,difuso_evolutivo,poblacion,"[40, 60, 80]","aleatoria, 14 configuraciones"
4,difuso_evolutivo,generaciones,"[60, 100, 140]","aleatoria, 14 configuraciones"
5,difuso_evolutivo,prob_mutacion,"[0.05, 0.15, 0.3]","aleatoria, 14 configuraciones"
6,difuso_evolutivo,prob_cruce,"[0.7, 0.9]","aleatoria, 14 configuraciones"
7,difuso_evolutivo,lambda_complejidad,"[0.0, 0.0005]","aleatoria, 14 configuraciones"


**Presupuesto de cómputo del difuso-evolutivo** (medido en el E2, Apple M1, un núcleo):

- un entrenamiento con la configuración máxima (15 reglas, población 80, 140 generaciones) tarda 284 s;
- entrenamientos previstos: 42 de búsqueda (14 × 3 ventanas), 27 de sensibilidad (9 × 3) y 15 de semillas (5 × 3), 84 en total;
- cota superior: 6.6 h en un núcleo, menos de 1 h en paralelo con 8 procesos.

**Criterio de parada:** número fijo de generaciones (no hay parada temprana), así el costo es predecible y comparable entre configuraciones.

## 7. Referencias en validación

Cada configuración se entrena con la partición de cada ventana y se valida con el año completo. La mejor configuración de cada modelo es la de mayor AUC-PR medio en las tres ventanas.

In [9]:
registro, t_ini = [], time.time()
for nombre, rejilla in REJILLAS.items():
    for hp in rejilla:
        for v, (a, b) in VENTANAS_VAL.items():
            tr, va = particion(a), tramo(dev, a, b)
            m = pipeline(nombre, **hp)
            t0 = time.time(); m.fit(tr[VARIABLES], tr.riesgoso); t_fit = time.time() - t0
            p = m.predict_proba(va[VARIABLES])[:, 1]
            registro.append({"fecha_hora": pd.Timestamp.now().isoformat(timespec="seconds"),
                             "modelo": nombre, "config": json.dumps(hp), "semilla": SEMILLA,
                             "ventana": v, "n_entrenamiento": len(tr),
                             "auc_pr": average_precision_score(va.riesgoso, p),
                             "auc_roc": roc_auc_score(va.riesgoso, p),
                             "prevalencia": va.riesgoso.mean(), "tiempo_s": t_fit})
registro = pd.DataFrame(registro)
registro.to_csv("registro_experimentos_e3.csv", index=False)
print(f"{len(registro)} entrenamientos en {time.time() - t_ini:.0f} s")
resumen = (registro.groupby(["modelo", "config"]).auc_pr.mean().rename("auc_pr_medio")
           .reset_index().sort_values(["modelo", "auc_pr_medio"], ascending=[True, False]))
resumen

30 entrenamientos en 84 s


,modelo,config,auc_pr_medio
0,gboosting,"{""learning_rate"": 0.05, ""max_leaf_nodes"": 15}",0.157777
1,gboosting,"{""learning_rate"": 0.05, ""max_leaf_nodes"": 31}",0.156007
3,gboosting,"{""learning_rate"": 0.1, ""max_leaf_nodes"": 15}",0.155672
2,gboosting,"{""learning_rate"": 0.05, ""max_leaf_nodes"": 63}",0.154862
5,gboosting,"{""learning_rate"": 0.1, ""max_leaf_nodes"": 63}",0.152688
4,gboosting,"{""learning_rate"": 0.1, ""max_leaf_nodes"": 31}",0.151895
6,logistica,"{""C"": 0.01}",0.148555
8,logistica,"{""C"": 1.0}",0.148554
7,logistica,"{""C"": 0.1}",0.148553
9,mayoritaria,{},0.040979


In [10]:
mejores = resumen.groupby("modelo").head(1).set_index("modelo")
sel = registro.merge(mejores.reset_index()[["modelo", "config"]])
tabla_val = sel.pivot(index="modelo", columns="ventana", values="auc_pr")
tabla_val["media"] = tabla_val.mean(axis=1)
tabla_val["caída 2023→2025 (%)"] = 100 * (tabla_val["2023"] - tabla_val["2025"]) / tabla_val["2023"]
tabla_val["AUC-ROC medio"] = sel.groupby("modelo").auc_roc.mean()
tabla_val["tiempo medio (s)"] = sel.groupby("modelo").tiempo_s.mean()
tabla_val["config"] = mejores.config
tabla_val.loc[["mayoritaria", "logistica", "gboosting"]].round(4)

ventana,2023,2024,2025,media,caída 2023→2025 (%),AUC-ROC medio,tiempo medio (s),config
modelo,,,,,,,,
mayoritaria,0.0389,0.0414,0.0427,0.0410,-9.8336,0.5000,0.0449,{}
logistica,0.1509,0.1433,0.1515,0.1486,-0.3740,0.7693,1.0309,"{""C"": 0.01}"
gboosting,0.1569,0.1537,0.1627,0.1578,-3.7082,0.7724,4.3969,"{""learning_rate"": 0.05, ""max_leaf_nodes"": 15}"


## 8. Reducción de variables por correlación

Para cada ventana calculamos la correlación de Spearman en el entrenamiento. Recorremos las variables en el orden del diccionario y descartamos una si su correlación absoluta con alguna ya conservada alcanza 0,85. Comparamos el mismo gradient boosting, con la configuración seleccionada en validación, antes y después del filtro.

In [11]:
comparacion = []
for ventana, (inicio, fin) in VENTANAS_VAL.items():
    entrenamiento = particion(inicio)
    validacion = tramo(dev, inicio, fin)
    correlacion = entrenamiento[VARIABLES].corr(method="spearman").abs()
    conservadas = []
    descartadas = []
    for variable in VARIABLES:
        if correlacion.loc[variable, conservadas].ge(0.85).any():
            descartadas.append(variable)
        else:
            conservadas.append(variable)
    modelo = pipeline("gboosting", **json.loads(mejores.loc["gboosting", "config"]))
    modelo.fit(entrenamiento[conservadas], entrenamiento.riesgoso)
    puntajes = modelo.predict_proba(validacion[conservadas])[:, 1]
    original = tabla_val.loc["gboosting", ventana]
    reducida = average_precision_score(validacion.riesgoso, puntajes)
    comparacion.append({"ventana": ventana, "variables": len(conservadas),
                        "descartadas": ", ".join(descartadas), "AP completa": original,
                        "AP reducida": reducida, "diferencia": reducida - original})
reduccion = pd.DataFrame(comparacion).set_index("ventana")
reduccion.round(5)

,variables,descartadas,AP completa,AP reducida,diferencia
ventana,,,,,
2023,21,n_subsistemas,0.15691,0.15840,0.00149
2024,21,n_subsistemas,0.15370,0.15310,-0.00060
2025,21,n_subsistemas,0.16273,0.16387,0.00115


Al conservar 21 entradas, la AUC-PR media pasó de 0,15778 a 0,15846. El filtro retiró el número de subsistemas en las tres ventanas. La diferencia fue positiva en 2023 y 2025 y negativa en 2024. Mantenemos la referencia completa para el contraste previsto.

## 9. Resumen

- La prueba de 2026 quedó reservada; solo se conocen su tamaño y su huella.
- Todas las transformaciones viven dentro de un `Pipeline` y se ajustan con el entrenamiento de cada ventana; la partición deja 90 días de hueco.
- Las métricas, las pruebas y los espacios de hiperparámetros quedan declarados antes de entrenar el modelo propuesto, que se entrena y ajusta en el Entregable 4.
- El registro de esta corrida queda en `registro_experimentos_e3.csv` y el resumen en `resultados_e3.json`.

In [12]:
salida = {"huella_prueba": HUELLA_PRUEBA, "n_prueba": N_PRUEBA, "limpieza": limpieza,
          "particion": tabla_particion.reset_index().to_dict("records"),
          "monotonia_experiencia": mono,
          "atipicos_pct_max": float(pasos["atipicos_%_entr2025"].max()),
          "mejores": {k: json.loads(v) for k, v in mejores.config.items()},
          "validacion": tabla_val.drop(columns="config").reset_index().to_dict("records"),
          "n_entrenamientos": len(registro), "segundos_total": float(registro.tiempo_s.sum()),
          "versiones": {"python": platform.python_version(), "pandas": pd.__version__,
                        "numpy": np.__version__, "scikit-learn": sklearn.__version__}}
with open("resultados_e3.json", "w") as f:
    json.dump(salida, f, ensure_ascii=False, indent=1, default=float)
print(json.dumps(salida["validacion"], ensure_ascii=False, indent=1, default=float))

[
 {
  "modelo": "gboosting",
  "2023": 0.1569078415039057,
  "2024": 0.15369554673528615,
  "2025": 0.1627262511134681,
  "media": 0.15777654645088665,
  "caída 2023→2025 (%)": -3.7081700658138206,
  "AUC-ROC medio": 0.7723882220468937,
  "tiempo medio (s)": 4.396894375483195
 },
 {
  "modelo": "logistica",
  "2023": 0.15089932110821047,
  "2024": 0.14330104567755786,
  "2025": 0.15146371708461845,
  "media": 0.14855469462346227,
  "caída 2023→2025 (%)": -0.3740215477863198,
  "AUC-ROC medio": 0.7692743275408223,
  "tiempo medio (s)": 1.030877987543742
 },
 {
  "modelo": "mayoritaria",
  "2023": 0.038865298419438546,
  "2024": 0.041385079933595637,
  "2025": 0.04268714011516315,
  "media": 0.040979172822732446,
  "caída 2023→2025 (%)": -9.833558086905366,
  "AUC-ROC medio": 0.5,
  "tiempo medio (s)": 0.04493109385172526
 }
]
